# Preprocessing: Cleaning, Lemmatization, and Stopwords removal
Tailored for English Netflix reviews with ASCII cleaning and WordNet Lemmatization.

In [11]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download NLTK requirements
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...


True

In [12]:
# Load data
df = pd.read_csv('../Output/Netflix_Review_Raw.csv', encoding='utf-8')
df = df.dropna(subset=['content'])
df.head()

,reviewId,userName,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,9e918801-fc09-4fbd-b577-c1623b466b18,A***********r,horrible company with an even worse app,1,0,9.59.0 build 8 63972,2026-03-30 21:36:35,NaN,NaN,9.59.0 build 8 63972
1,785815e4-2645-473f-85f7-7fd54e375411,A***********r,enjoyed,5,0,NaN,2026-03-30 21:31:29,NaN,NaN,NaN
2,68aab1b0-801f-43b4-be84-cab1d9245868,A***********r,संजसद्बुकद्दूक्ष्मस्लास्म,5,0,NaN,2026-03-30 21:27:39,NaN,NaN,NaN
3,02467eab-9443-4dba-a415-f367fdf22ab4,A***********r,where is steel ball run netflix? where is it? ...,1,0,NaN,2026-03-30 21:26:51,NaN,NaN,NaN
4,9d984e77-9bd9-40ba-a5bb-be3e265b3c81,A***********r,no steel ball run ep2,1,0,9.59.0 build 8 63972,2026-03-30 21:26:39,NaN,NaN,9.59.0 build 8 63972


## 1. Aggressive Cleaning
Removes non-ASCII (Hindi/Emoji), links, punctuation, and numbers.

In [13]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Remove non-ASCII characters
    text = text.encode("ascii", "ignore").decode()
    # Remove links
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove extra whitespace
    text = " ".join(text.split())
    return text

df['cleaned_content'] = df['content'].apply(clean_text)
df = df[df['cleaned_content'] != ""]

## 2. Lemmatization and Stopword Removal
Lemmatization reduces words to their base form (e.g., 'watching' -> 'watch').

In [14]:
lemmatizer = WordNetLemmatizer()
en_stopwords = set(stopwords.words('english'))

# Custom stopwords adjusted for lemmatized forms
custom_stopwords = {
    'netflix', 'app', 'application', 'update', 'watch', 'movie', 'film', 'show', 'video', 
    'stream', 'streaming', 'content', 'service', 'good', 'great', 'nice', 'bad', 'best', 
    'worst', 'horrible', 'awesome', 'excellent', 'please', 'pls', 'thanks', 'thank', 
    'get', 'see', 'make', 'even', 'still', 'already', 'though', 'would', 'could', 'should', 
    'try', 'fix', 'issue', 'problem', 'error', 'wait', 'much', 'many', 'lot', 'give', 
    'take', 'put', 'keep', 'come', 'go', 'use', 'user', 'review', 'rating', 'star', 
    'ad', 'money', 'pay', 'paid', 'subscription', 'subscribe', 'account', 'login', 
    'phone', 'device', 'screen', 'tv', 'cast'
}

all_stopwords = en_stopwords.union(custom_stopwords)

def lemmatize_and_filter(text):
    # Tokenization
    tokens = word_tokenize(text)
    
    # Lemmatization (converting words to base form)
    # Note: Default is noun, for better results POS tagging is needed, 
    # but this is usually sufficient for general review analysis.
    lemmas = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Stopword Removal
    filtered = [word for word in lemmas if word not in all_stopwords and len(word) > 1]
    
    return filtered

# Apply process
df['tokenized_filtered'] = df['cleaned_content'].apply(lemmatize_and_filter)
df['final_content'] = df['tokenized_filtered'].apply(lambda x: " ".join(x))

# Final cleanup of empty results
df = df[df['final_content'] != ""]

df[['content', 'final_content']].head()

,content,final_content
0,horrible company with an even worse app,company worse
1,enjoyed,enjoyed
3,where is steel ball run netflix? where is it? ...,steel ball run like tormenting steel ball run ...
4,no steel ball run ep2,steel ball run ep
5,Steel Ball Run.,steel ball run


## 3. Save Results

In [15]:
output_path = '../Output/Netflix_Review_Preprocessed.csv'
df.to_csv(output_path, index=False)
print(f"Success! Data with Lemmatization saved to {output_path}")
print(f"Final count: {len(df)} reviews.")

Success! Data with Lemmatization saved to ../Output/Netflix_Review_Preprocessed.csv
Final count: 82441 reviews.
